# Text Feature Engineering Assignment — Complete Implementation

This notebook covers the full assignment workflow:

1. Collect real-world product reviews
2. Preprocess text
3. Build vocabulary
4. Create One-Hot, Bag of Words, and TF-IDF features
5. Compare sparse matrices
6. Answer real-world theory questions
7. Train a mini sentiment classifier using BoW and TF-IDF

> Recommended: **Flipkart review pages** are usually easier than Amazon for student assignments.
> This notebook includes both:
> - a **web scraping pipeline** for real product review pages
> - a **CSV fallback pipeline** if you already collected reviews manually

In [ ]:
# Optional installs (run only if needed)
# !pip install requests beautifulsoup4 pandas numpy scikit-learn matplotlib seaborn lxml nltk

In [15]:
import re
import string
import math
import warnings
from collections import Counter

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

## 1) Load the dataset from `review.csv`

This notebook is designed to use an existing CSV file named `review.csv`.

Required column:
- `review_text`

Optional column:
- `rating` (recommended for the sentiment classification task)

Place `review.csv` in the same folder as this notebook, or update the file path in the next code cell.


In [16]:
# CSV input
CSV_PATH = "review.csv"   # change this if needed

def load_reviews_csv(csv_path=CSV_PATH):
    df = pd.read_csv(csv_path)
    if "review_text" not in df.columns:
        raise ValueError("CSV must contain a 'review_text' column.")
    df = df.dropna(subset=["review_text"]).copy()
    df["review_text"] = df["review_text"].astype(str)
    return df

# Example:
# reviews_df = load_reviews_csv("review.csv")
# reviews_df.head()


## 2) Data cleaning and preprocessing
Required in assignment:
- lowercase
- tokenization
- remove punctuation

Optional:
- remove stopwords
- lemmatization

This notebook keeps the optional steps configurable.

In [17]:
import nltk

# Uncomment the first time you run
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

STOPWORDS = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text, remove_stopwords=True, do_lemmatize=True):
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)             # remove URLs
    text = re.sub(r"<.*?>", " ", text)                           # remove HTML tags
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\d+", " ", text)                            # optional: remove numbers
    tokens = text.split()

    if remove_stopwords:
        tokens = [t for t in tokens if t not in STOPWORDS]

    if do_lemmatize:
        tokens = [lemmatizer.lemmatize(t) for t in tokens]

    tokens = [t for t in tokens if len(t) > 1]                   # remove tiny tokens
    return " ".join(tokens)

def add_preprocessed_columns(df):
    df = df.copy()
    df["review_text"] = df["review_text"].astype(str).str.strip()
    df = df[df["review_text"].str.len() > 0].copy()
    df["clean_text"] = df["review_text"].apply(preprocess_text)
    df["tokens"] = df["clean_text"].str.split()
    return df.reset_index(drop=True)

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/srikanthkuricheti/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/srikanthkuricheti/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/srikanthkuricheti/nltk_data...


In [ ]:
# Load the dataset first
# reviews_df = load_reviews_csv("review.csv")

# For assignment execution, uncomment the above line first.


In [19]:
# Example execution cell after loading the dataset
reviews_df = load_reviews_csv("review.csv")
reviews_df = add_preprocessed_columns(reviews_df)
reviews_df[["review_text", "clean_text", "tokens"]].head()


,review_text,clean_text,tokens
0,Worst purchase ever,worst purchase ever,"[worst, purchase, ever]"
1,Bad experience,bad experience,"[bad, experience]"
2,"Great product, really loved it",great product really loved,"[great, product, really, loved]"
3,Very satisfied with the purchase,satisfied purchase,"[satisfied, purchase]"
4,Product stopped working,product stopped working,"[product, stopped, working]"


## 3) Vocabulary creation and top frequent words

In [20]:
def build_manual_vocabulary(token_lists):
    vocab_counter = Counter()
    for tokens in token_lists:
        vocab_counter.update(tokens)
    vocab = sorted(vocab_counter.keys())
    return vocab, vocab_counter

def show_top_frequent_words(df, top_n=20):
    vocab, vocab_counter = build_manual_vocabulary(df["tokens"])
    print("Vocabulary size:", len(vocab))
    print("\nTop frequent words:")
    top_words = pd.DataFrame(vocab_counter.most_common(top_n), columns=["word", "count"])
    display(top_words)
    return vocab, vocab_counter, top_words

# Example:
# vocab, vocab_counter, top_words = show_top_frequent_words(reviews_df, top_n=20)

## 4) Feature engineering
We will create:
1. **One Hot Encoding** (document-level)
2. **Bag of Words** using `CountVectorizer`
3. **TF-IDF** using `TfidfVectorizer`

In [21]:
def document_level_one_hot(token_lists):
    mlb = MultiLabelBinarizer()
    ohe_matrix = mlb.fit_transform(token_lists)
    feature_names = list(mlb.classes_)
    return ohe_matrix, feature_names, mlb

def bag_of_words_features(corpus, max_features=None):
    vectorizer = CountVectorizer(max_features=max_features)
    X_bow = vectorizer.fit_transform(corpus)
    return X_bow, vectorizer.get_feature_names_out(), vectorizer

def tfidf_features(corpus, max_features=None):
    vectorizer = TfidfVectorizer(max_features=max_features)
    X_tfidf = vectorizer.fit_transform(corpus)
    return X_tfidf, vectorizer.get_feature_names_out(), vectorizer

In [23]:
# Example feature creation
X_ohe, ohe_features, ohe_encoder = document_level_one_hot(reviews_df["tokens"])
X_bow, bow_features, bow_vectorizer = bag_of_words_features(reviews_df["clean_text"])
X_tfidf, tfidf_features_names, tfidf_vectorizer = tfidf_features(reviews_df["clean_text"])

print("OHE shape:", X_ohe.shape)
print("BoW shape:", X_bow.shape)
print("TF-IDF shape:", X_tfidf.shape)

OHE shape: (120, 39)
BoW shape: (120, 39)
TF-IDF shape: (120, 39)


## 5) Sparse matrix analysis
We will:
- print matrix shapes
- calculate sparsity
- explain the result

In [25]:
from scipy import sparse

def calculate_sparsity(matrix):
    # Works for both dense numpy arrays and scipy sparse matrices
    if sparse.issparse(matrix):
        total = matrix.shape[0] * matrix.shape[1]
        non_zero = matrix.nnz
    else:
        total = matrix.size
        non_zero = np.count_nonzero(matrix)

    zero_count = total - non_zero
    sparsity = (zero_count / total) * 100
    return {
        "shape": matrix.shape,
        "total_elements": total,
        "non_zero_elements": int(non_zero),
        "zero_elements": int(zero_count),
        "sparsity_percent": round(sparsity, 2),
    }

def compare_sparsity(X_ohe, X_bow, X_tfidf):
    rows = []
    for name, matrix in {
        "One Hot Encoding": X_ohe,
        "Bag of Words": X_bow,
        "TF-IDF": X_tfidf,
    }.items():
        stats = calculate_sparsity(matrix)
        stats["representation"] = name
        rows.append(stats)

    cols = ["representation", "shape", "total_elements", "non_zero_elements", "zero_elements", "sparsity_percent"]
    return pd.DataFrame(rows)[cols]

# Example:
sparsity_df = compare_sparsity(X_ohe, X_bow, X_tfidf)
display(sparsity_df)

,representation,shape,total_elements,non_zero_elements,zero_elements,sparsity_percent
0,One Hot Encoding,"(120, 39)",4680,302,4378,93.55
1,Bag of Words,"(120, 39)",4680,302,4378,93.55
2,TF-IDF,"(120, 39)",4680,302,4378,93.55


## 6) TF-IDF: most important words
High TF-IDF means:
- the word appears frequently in a specific document
- but not in too many documents overall

So very common words across many reviews receive lower weight.

In [26]:
def top_tfidf_words_per_document(X_tfidf, feature_names, original_df, top_n=10, sample_docs=5):
    feature_names = np.array(feature_names)
    results = []

    num_docs = min(sample_docs, X_tfidf.shape[0])

    for i in range(num_docs):
        row = X_tfidf[i].toarray().flatten()
        top_idx = row.argsort()[::-1][:top_n]
        top_terms = [(feature_names[j], round(float(row[j]), 4)) for j in top_idx if row[j] > 0]

        results.append({
            "doc_id": i,
            "review_text": original_df.iloc[i]["review_text"][:200],
            "top_tfidf_terms": top_terms
        })

    return pd.DataFrame(results)

# Example:
tfidf_top_df = top_tfidf_words_per_document(X_tfidf, tfidf_features_names, reviews_df, top_n=10, sample_docs=5)
display(tfidf_top_df)

,doc_id,review_text,top_tfidf_terms
0,0,Worst purchase ever,"[(worst, 0.6129), (ever, 0.6129), (purchase, 0.4986)]"
1,1,Bad experience,"[(bad, 0.737), (experience, 0.6758)]"
2,2,"Great product, really loved it","[(great, 0.5485), (really, 0.5485), (loved, 0.4908), (product, 0.3967)]"
3,3,Very satisfied with the purchase,"[(satisfied, 0.7657), (purchase, 0.6432)]"
4,4,Product stopped working,"[(working, 0.6473), (stopped, 0.6473), (product, 0.4026)]"


## 7) Comparison table: OHE vs BoW vs TF-IDF

In [27]:
comparison_table = pd.DataFrame([
    {
        "Method": "One Hot Encoding",
        "What it stores": "Only presence/absence of words",
        "Captures frequency?": "No",
        "Captures importance?": "No",
        "Typical use": "Very simple baseline or categorical word presence",
        "Main limitation": "Ignores frequency and semantic meaning",
    },
    {
        "Method": "Bag of Words",
        "What it stores": "Word counts in each document",
        "Captures frequency?": "Yes",
        "Captures importance?": "Partly",
        "Typical use": "Baseline text classification and classical ML",
        "Main limitation": "Ignores context and synonym meaning",
    },
    {
        "Method": "TF-IDF",
        "What it stores": "Weighted word importance",
        "Captures frequency?": "Yes",
        "Captures importance?": "Yes",
        "Typical use": "Search, ranking, document similarity, classification",
        "Main limitation": "Still ignores deep semantics and word order",
    },
])

comparison_table

,Method,What it stores,Captures frequency?,Captures importance?,Typical use,Main limitation
0,One Hot Encoding,Only presence/absence of words,No,No,Very simple baseline or categorical word presence,Ignores frequency and semantic meaning
1,Bag of Words,Word counts in each document,Yes,Partly,Baseline text classification and classical ML,Ignores context and synonym meaning
2,TF-IDF,Weighted word importance,Yes,Yes,"Search, ranking, document similarity, classification",Still ignores deep semantics and word order


## 8) Real-world theory answers
Use these in your short report.

In [28]:
theory_answers = {
    "Why BoW fails in semantic understanding":
        "BoW treats words as independent tokens and ignores meaning, context, and word order. "
        "For example, 'good' and 'excellent' are semantically similar but appear as completely different features. "
        "Likewise, 'not good' and 'good' can look similar because BoW does not properly capture negation.",

    "When to use BoW in industry":
        "Use BoW when you need a fast baseline, interpretable features, small to medium datasets, or simple classification tasks "
        "such as spam detection, review classification, or ticket categorization.",

    "When to use TF-IDF in industry":
        "Use TF-IDF when important keywords matter more than raw counts, such as search systems, information retrieval, document ranking, "
        "FAQ matching, and traditional ML pipelines for text classification.",

    "Limitations of TF-IDF":
        "TF-IDF ignores word order, does not understand context, cannot capture semantic similarity well, struggles with synonyms, "
        "and does not adapt to changing meaning across domains. It also produces high-dimensional sparse features."
}

for k, v in theory_answers.items():
    print(f"\n{k}:\n{v}\n")


Why BoW fails in semantic understanding:
BoW treats words as independent tokens and ignores meaning, context, and word order. For example, 'good' and 'excellent' are semantically similar but appear as completely different features. Likewise, 'not good' and 'good' can look similar because BoW does not properly capture negation.


When to use BoW in industry:
Use BoW when you need a fast baseline, interpretable features, small to medium datasets, or simple classification tasks such as spam detection, review classification, or ticket categorization.


When to use TF-IDF in industry:
Use TF-IDF when important keywords matter more than raw counts, such as search systems, information retrieval, document ranking, FAQ matching, and traditional ML pipelines for text classification.


Limitations of TF-IDF:
TF-IDF ignores word order, does not understand context, cannot capture semantic similarity well, struggles with synonyms, and does not adapt to changing meaning across domains. It also produ

## 9) Mini use case: sentiment classification

To do this well, your dataset should include a `rating` column.

We will derive labels like this:
- rating >= 4  → positive
- rating <= 2  → negative
- rating == 3  → neutral (dropped for binary classification)

If your data has no ratings, you can manually label a smaller subset.

In [29]:
def create_binary_sentiment_labels(df):
    if "rating" not in df.columns:
        raise ValueError("Need a 'rating' column for binary sentiment labeling.")

    df = df.copy()
    df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
    df = df.dropna(subset=["rating"]).copy()

    def map_sentiment(r):
        if r >= 4:
            return "positive"
        elif r <= 2:
            return "negative"
        return "neutral"

    df["sentiment"] = df["rating"].apply(map_sentiment)
    df = df[df["sentiment"] != "neutral"].copy()
    return df.reset_index(drop=True)

In [30]:
def train_and_evaluate_models(df, text_column="clean_text", label_column="sentiment"):
    df = df.copy()

    # Defensive fallback: build sentiment labels from rating if needed.
    if label_column not in df.columns and "rating" in df.columns:
        df = create_binary_sentiment_labels(df)

    if text_column not in df.columns:
        raise ValueError(f"Missing required text column: '{text_column}'")
    if label_column not in df.columns:
        raise ValueError(f"Missing required label column: '{label_column}'. Add labels or provide a 'rating' column.")

    X_text = df[text_column].astype(str)
    y = df[label_column].astype(str)

    if y.nunique() < 2:
        raise ValueError("Need at least two sentiment classes to train the model.")

    X_train_text, X_test_text, y_train, y_test = train_test_split(
        X_text, y, test_size=0.2, random_state=42, stratify=y
    )

    # BoW features
    bow_vectorizer = CountVectorizer()
    X_train_bow = bow_vectorizer.fit_transform(X_train_text)
    X_test_bow = bow_vectorizer.transform(X_test_text)

    # TF-IDF features
    tfidf_vectorizer = TfidfVectorizer()
    X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_text)
    X_test_tfidf = tfidf_vectorizer.transform(X_test_text)

    results = []

    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000),
        "Naive Bayes": MultinomialNB()
    }

    for model_name, model in models.items():
        # BoW
        model.fit(X_train_bow, y_train)
        pred_bow = model.predict(X_test_bow)
        acc_bow = accuracy_score(y_test, pred_bow)

        results.append({
            "Model": model_name,
            "Features": "BoW",
            "Accuracy": round(acc_bow, 4)
        })

        # TF-IDF
        model.fit(X_train_tfidf, y_train)
        pred_tfidf = model.predict(X_test_tfidf)
        acc_tfidf = accuracy_score(y_test, pred_tfidf)

        results.append({
            "Model": model_name,
            "Features": "TF-IDF",
            "Accuracy": round(acc_tfidf, 4)
        })

    results_df = pd.DataFrame(results).sort_values(["Model", "Accuracy"], ascending=[True, False]).reset_index(drop=True)
    return results_df

In [31]:
# Example sentiment pipeline
sentiment_df = create_binary_sentiment_labels(reviews_df)
sentiment_results = train_and_evaluate_models(sentiment_df)
display(sentiment_results)

,Model,Features,Accuracy
0,Logistic Regression,BoW,0.9583
1,Logistic Regression,TF-IDF,0.9583
2,Naive Bayes,BoW,0.9583
3,Naive Bayes,TF-IDF,0.9583


## 10) Save outputs for submission
This section saves:
- cleaned dataset
- feature matrices (optional)
- report-ready comparison tables

In [32]:
def save_assignment_outputs(df, output_dir="assignment_outputs"):
    import os
    os.makedirs(output_dir, exist_ok=True)

    df.to_csv(f"{output_dir}/clean_reviews.csv", index=False)
    comparison_table.to_csv(f"{output_dir}/comparison_table.csv", index=False)

    print(f"Saved files to: {output_dir}")

# Example:
save_assignment_outputs(reviews_df)

Saved files to: assignment_outputs


## 11) Final execution block

Run this after loading `review.csv` and making sure `reviews_df` exists.


In [36]:
# ==============================
# FINAL EXECUTION BLOCK
# ==============================

# Step 1: Load data
reviews_df = load_reviews_csv("review.csv")

# Step 2: Preprocess
reviews_df = add_preprocessed_columns(reviews_df)

# Step 3: Vocabulary
vocab, vocab_counter, top_words = show_top_frequent_words(reviews_df, top_n=20)

# Step 4: Features
X_ohe, ohe_features, ohe_encoder = document_level_one_hot(reviews_df["tokens"])
X_bow, bow_features, bow_vectorizer = bag_of_words_features(reviews_df["clean_text"])
X_tfidf, tfidf_features_names, tfidf_vectorizer = tfidf_features(reviews_df["clean_text"])

# Step 5: Shapes and sparsity
print("OHE shape:", X_ohe.shape)
print("BoW shape:", X_bow.shape)
print("TF-IDF shape:", X_tfidf.shape)
sparsity_df = compare_sparsity(X_ohe, X_bow, X_tfidf)
display(sparsity_df)

# Step 6: Important TF-IDF words
tfidf_top_df = top_tfidf_words_per_document(X_tfidf, tfidf_features_names, reviews_df, top_n=10, sample_docs=5)
display(tfidf_top_df)

# Step 7: Sentiment classification (only if rating exists)
if "rating" in reviews_df.columns:
    sentiment_df = create_binary_sentiment_labels(reviews_df)
    sentiment_results = train_and_evaluate_models(sentiment_df)
    display(sentiment_results)
else:
    print("No 'rating' column found. Add ratings if you want the sentiment classification part.")

# Step 8: Save outputs
output_paths = save_assignment_outputs(reviews_df)
print(output_paths)


Vocabulary size: 39

Top frequent words:


,word,count
0,product,29
1,quality,24
2,experience,16
3,loved,16
4,purchase,15
5,bad,12
6,great,11
7,really,11
8,poor,9
9,disappointed,9


OHE shape: (120, 39)
BoW shape: (120, 39)
TF-IDF shape: (120, 39)


,representation,shape,total_elements,non_zero_elements,zero_elements,sparsity_percent
0,One Hot Encoding,"(120, 39)",4680,302,4378,93.55
1,Bag of Words,"(120, 39)",4680,302,4378,93.55
2,TF-IDF,"(120, 39)",4680,302,4378,93.55


,doc_id,review_text,top_tfidf_terms
0,0,Worst purchase ever,"[(worst, 0.6129), (ever, 0.6129), (purchase, 0.4986)]"
1,1,Bad experience,"[(bad, 0.737), (experience, 0.6758)]"
2,2,"Great product, really loved it","[(great, 0.5485), (really, 0.5485), (loved, 0.4908), (product, 0.3967)]"
3,3,Very satisfied with the purchase,"[(satisfied, 0.7657), (purchase, 0.6432)]"
4,4,Product stopped working,"[(working, 0.6473), (stopped, 0.6473), (product, 0.4026)]"


,Model,Features,Accuracy
0,Logistic Regression,BoW,0.9583
1,Logistic Regression,TF-IDF,0.9583
2,Naive Bayes,BoW,0.9583
3,Naive Bayes,TF-IDF,0.9583


Saved files to: assignment_outputs
None


## 12) Short conclusion for report

You can reuse this in your 1–2 page report:

- One Hot Encoding is the simplest representation and shows only whether a word appears or not.
- Bag of Words improves this by storing word frequency, which helps classical machine learning models.
- TF-IDF is more informative because it gives higher weight to important words that are frequent in a specific document but not common across all documents.
- All three approaches usually produce sparse matrices with many zeros because each review contains only a small fraction of the total vocabulary.
- Sparse representations save memory in optimized libraries, but very large vocabularies can still become inefficient in production systems.
- For sentiment classification, TF-IDF often performs slightly better than BoW because it reduces the impact of very common words.